In [ ]:
#1: Cai thu vien
# !pip install soundfile pandas numpy librosa

In [ ]:
#2: Imports
import os
import re
import gc
import io
import json
import shutil
import numpy as np
import pandas as pd
import soundfile as sf
from glob import glob

In [ ]:
#3: Config duong dan

VOCAB_PATH    = r'D:\ViMD-data-prepare-for-Whisper-main\vocab.json'
TRAIN_PARQUET = r'D:\ViMD-data-prepare-for-Whisper-main\data\train\*.parquet'
VAL_PARQUET   = r'D:\ViMD-data-prepare-for-Whisper-main\data\val\*.parquet'
TEST_PARQUET  = r'D:\ViMD-data-prepare-for-Whisper-main\data\test\*.parquet'
OUTPUT_DIR    = r'D:\ViMD-data-prepare-for-main\vimd-processed'

MIN_DURATION  = 0.5
MAX_DURATION  = 50.0

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'Output dir: {OUTPUT_DIR}')

# Kiem tra vocab
if not os.path.exists(VOCAB_PATH):
    print('Chua co vocab.json!')
    print('Tai ve tai: https://huggingface.co/nguyenvulebinh/wav2vec2-base-vietnamese-250h/resolve/main/vocab.json')
    print(f'Luu vao: {VOCAB_PATH}')
else:
    print('vocab.json found!')

# Kiem tra parquet files
train_files = glob(TRAIN_PARQUET)
val_files   = glob(VAL_PARQUET)
test_files  = glob(TEST_PARQUET)
print(f'Train parquet: {len(train_files)} files')
print(f'Val   parquet: {len(val_files)} files')
print(f'Test  parquet: {len(test_files)} files')

In [ ]:
#4: Load & Fix Vocab (vocab=110)
with open(VOCAB_PATH, encoding='utf-8') as f:
    vocab = json.load(f)

# Bo <s> va </s> -> vocab=110
vocab_clean = {k: v for k, v in vocab.items() if k not in ['<s>', '</s>']}
print(f'Vocab size: {len(vocab_clean)}')  # phai ra 110

# Luu vocab_clean de dung tren Kaggle
vocab_clean_path = os.path.join(OUTPUT_DIR, 'vocab_clean.json')
with open(vocab_clean_path, 'w', encoding='utf-8') as f:
    json.dump(vocab_clean, f, ensure_ascii=False, indent=2)
print(f'Saved vocab_clean.json')

# Tạo mapping
token2id = vocab_clean
id2token = {v: k for k, v in vocab_clean.items()}
PAD_ID   = token2id['<pad>']
UNK_ID   = token2id['<unk>']
SEP_ID   = token2id['|']

print(f'PAD={PAD_ID} | UNK={UNK_ID} | SEP={SEP_ID}')

In [ ]:
#5: Functions xu ly

# Normalize audio
def normalize_audio(array):
    mean = np.mean(array)
    std  = np.std(array)
    if std > 0:
        array = (array - mean) / std
    return array.astype(np.float32)

# Normalize text
VIETNAMESE_CHARS = (
    r'àáảãạăắằẳẵặâấầẩẫậ'
    r'đèéẻẽẹêếềểễệ'
    r'ìíỉĩịòóỏõọôốồổỗộơớờởỡợ'
    r'ùúủũụưứừửữựỳýỷỹỵ'
)
KEEP_PATTERN = re.compile(rf'[^\w\s{VIETNAMESE_CHARS}]', flags=re.UNICODE)

def normalize_text(text):
    text = text.lower()
    text = KEEP_PATTERN.sub('', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Tokenize text -> list of token ids
def tokenize(text):
    text  = normalize_text(text)
    chars = list(text)
    ids   = []
    for ch in chars:
        if ch == ' ':
            ids.append(SEP_ID)
        elif ch in token2id:
            ids.append(token2id[ch])
        else:
            ids.append(UNK_ID)
    return ids

In [ ]:
#6: Xử lý parquet -> npy
def process_parquet_files(parquet_pattern, output_prefix, split_name='train'):
    files = glob(parquet_pattern)
    if len(files) == 0:
        print(f'Khong tim thay file tai: {parquet_pattern}')
        return 0

    print(f'\nProcessing {split_name}: {len(files)} parquet files...')

    BATCH_SIZE  = 500
    all_iv      = []
    all_lb      = []
    skipped     = 0
    total       = 0
    saved_total = 0
    batch_idx   = 0

    def flush_batch():
        nonlocal batch_idx, saved_total
        if len(all_iv) == 0:
            return
        iv_path = f'{output_prefix}_iv_{batch_idx:04d}.npy'
        lb_path = f'{output_prefix}_lb_{batch_idx:04d}.npy'
        np.save(iv_path, np.array(all_iv, dtype=object))
        np.save(lb_path, np.array(all_lb, dtype=object))
        saved_total += len(all_iv)
        print(f'  Batch {batch_idx}: {len(all_iv)} samples | Total: {saved_total:,}')
        batch_idx += 1
        all_iv.clear()
        all_lb.clear()
        gc.collect()

    for file_idx, parquet_file in enumerate(files):
        print(f'\n  File {file_idx+1}/{len(files)}: {os.path.basename(parquet_file)}')
        df = pd.read_parquet(parquet_file)

        for i, row in df.iterrows():
            total += 1
            try:
                audio = row['audio']

                if isinstance(audio, dict) and 'bytes' in audio:
                    array, sr = sf.read(io.BytesIO(audio['bytes']))
                elif isinstance(audio, dict) and 'array' in audio:
                    array = np.array(audio['array'])
                    sr    = audio['sampling_rate']
                else:
                    skipped += 1
                    continue

                array = array.astype(np.float32)
                if len(array.shape) > 1:
                    array = array.mean(axis=1)

                if sr != 16000:
                    try:
                        import librosa
                        array = librosa.resample(array, orig_sr=sr, target_sr=16000)
                    except Exception:
                        skipped += 1
                        continue

                duration = len(array) / 16000
                if duration < MIN_DURATION or duration > MAX_DURATION:
                    skipped += 1
                    continue

                text = str(row.get('text', '')).strip()
                if not text:
                    skipped += 1
                    continue

                iv        = normalize_audio(array)
                label_ids = tokenize(text)
                label_ids = [x for x in label_ids if x < len(vocab_clean)]
                if len(label_ids) == 0:
                    skipped += 1
                    continue

                all_iv.append(iv)
                all_lb.append(np.array(label_ids, dtype=np.int32))

                if len(all_iv) >= BATCH_SIZE:
                    flush_batch()

            except Exception as e:
                skipped += 1
                if skipped <= 5:
                    print(f'    Skip {i}: {e}')

    # Flush batch cuối
    flush_batch()

    print(f'\n{split_name} hoan thanh!')
    print(f'  Total:   {total:,}')
    print(f'  Saved:   {saved_total:,}')
    print(f'  Skipped: {skipped:,}')
    print(f'  Batches: {batch_idx}')

    return saved_total

In [ ]:
#7: Xử lý TRAIN
n_train = process_parquet_files(
    TRAIN_PARQUET,
    os.path.join(OUTPUT_DIR, 'train'),
    split_name='train',
)
print(f'\nTrain : {n_train:,} samples')

In [ ]:
#8: Xử lý VAL
n_val = process_parquet_files(
    VAL_PARQUET,
    os.path.join(OUTPUT_DIR, 'val'),
    split_name='val',
)
print(f'\nVal : {n_val:,} samples')

In [ ]:
#9: Xử lý TEST
n_test = process_parquet_files(
    TEST_PARQUET,
    os.path.join(OUTPUT_DIR, 'test'),
    split_name='test',
)
print(f'\nTest : {n_test:,} samples')

In [ ]:
#10: Tổng Kết
print('\n' + '='*50)
print('Tổng Kết')
print('='*50)
print(f'  Train: {n_train:,} samples')
print(f'  Val:   {n_val:,} samples')
print(f'  Test:  {n_test:,} samples')
print(f'  Total: {n_train+n_val+n_test:,} samples')
print()
print(f'Files trong {OUTPUT_DIR}:')
for f in sorted(os.listdir(OUTPUT_DIR)):
    path = os.path.join(OUTPUT_DIR, f)
    if os.path.isfile(path):
        size = os.path.getsize(path) / 1e9
        print(f'  {f}: {size:.2f} GB')
print('='*50)

In [ ]:
#11: Test load lai de kiem tra
print('Test load lai...')

def test_load(split):
    iv_path = os.path.join(OUTPUT_DIR, f'{split}_iv_0000.npy')
    lb_path = os.path.join(OUTPUT_DIR, f'{split}_lb_0000.npy')

    if not os.path.exists(iv_path):
        print(f'  {split}: Khong tim thay {iv_path}')
        return

    iv = np.load(iv_path, allow_pickle=True)
    lb = np.load(lb_path, allow_pickle=True)

    print(f'\n[{split}] batch[0]: {len(iv):,} samples')
    print(f'  Sample[0] audio shape:  {iv[0].shape}')
    print(f'  Sample[0] duration:     {len(iv[0])/16000:.2f}s')
    print(f'  Sample[0] labels:       {lb[0]}')
    decoded = "".join(id2token.get(i, "?") for i in lb[0]).replace("|", " ")
    print(f'  Sample[0] decoded:      {decoded}')

test_load('train')
test_load('val')
test_load('test')